# Official data exploration - initial phase
Run from the project root. This notebook checks coverage and preprocessing only; it does not create player rankings. See `reports/data_audit.md` for all columns, nulls, and key checks.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from src.data import load_official_data, attach_match_context
from src.preprocessing import prepare_deliveries, normalize_playing_eleven

tables = load_official_data(Path.cwd())
pd.DataFrame({'table': list(tables), 'rows': [len(t) for t in tables.values()]})

In [ ]:
balls = prepare_deliveries(tables['ball_by_ball'])
context = attach_match_context(balls, tables['matches'], tables['competitions'])
assert len(context) == len(balls)
coverage = context.groupby(['match_format', 'season'], dropna=False).agg(
    deliveries=('match_id', 'size'), matches=('match_id', 'nunique')
).reset_index()
coverage, np.quantile(context.groupby('match_id').size(), [0.1, 0.5, 0.9])

In [ ]:
roster = normalize_playing_eleven(tables['playing_eleven'])
pd.Series({
    'deliveries': len(balls),
    'legal_delivery_unknown': int(balls.is_legal_delivery.isna().sum()),
    'pre_ball_legal_count_unknown': int(balls.legal_balls_before.isna().sum()),
    'roster_order_conflicts': int(roster.batting_order_conflict.sum()),
    'super_over_matches': int(context.loc[context.innings > 2, 'match_id'].nunique()),
    'matches_with_revised_target': int(tables['matches'].revised_target.notna().sum()),
})

## Interpretation limit
`extras1_runs` is a category code in this pack. Do not sum it as runs. Exact innings scores, required run rates, or bowling economy are withheld pending verification of extras encoding.